# AEMO SB3 online RL training notebook

Use this notebook to train an online SB3 policy directly on `AEMOBatteryTradingEnv`, with optional battery-size sweeps and rollout export for offline DT data generation.

In [ ]:
from pathlib import Path
import sys
from datetime import datetime

import polars as pl
from stable_baselines3.common.vec_env import DummyVecEnv

REPO_ROOT = Path.cwd()
sys.path.insert(0, str(REPO_ROOT / 'src'))

In [ ]:
from aemo_notebook_utils import (
    fetch_and_preprocess_aemo_data,
    make_aemo_env_fns,
    prepare_run_paths,
    resolve_battery_variants,
    train_sb3_model_on_aemo,
)
from decision import run_sb3_model_on_vec_env
from helper import flatten_episode_data

## 1. Training configuration

In [ ]:
REGION = 'SA1'
START_DATE = datetime.fromisoformat('2024-01-01')
END_DATE = datetime.fromisoformat('2024-02-01')
STEP_DURATION = 5 / 60
EPISODE_HOURS = 24
ACTION_MODE = 'multi_market'
DEGRADATION_MODE = 'real_world'
DEGRADATION_CHEMISTRY = 'LFP'
DEGRADATION_TEMPERATURE = 30.0

CACHE_DIR = REPO_ROOT / 'data' / 'aemo'
OUTPUT_DIR = REPO_ROOT / 'data' / 'aemo_sb3'
RUN_TAG = 'aemo_sb3'
SB3_ALGORITHM = 'PPO'

BATTERY_VARIANTS = [
    {'name': 'small', 'capacity_mwh': 2.0, 'max_power_mw': 1.0, 'init_soc_ratio': 0.5},
    {'name': 'medium', 'capacity_mwh': 10.0, 'max_power_mw': 5.0, 'init_soc_ratio': 0.5},
    {'name': 'large', 'capacity_mwh': 50.0, 'max_power_mw': 25.0, 'init_soc_ratio': 0.5},
]

EPISODES_PER_VARIANT = 2
DEFAULT_MODEL = True
TEST_TIMESTEPS = 20_000
TOTAL_TIMESTEPS = 200_000
N_TRIALS = 5
N_JOBS = 1
ROLLOUT_EPISODES_PER_VARIANT = 1
DETERMINISTIC_ROLLOUT = True

## 2. Fetch and cache AEMO data

In [ ]:
run_paths = prepare_run_paths(output_dir=OUTPUT_DIR, dataset_tag=RUN_TAG)
processed_data, processed_cache = fetch_and_preprocess_aemo_data(
    region=REGION,
    start_date=START_DATE,
    end_date=END_DATE,
    cache_dir=CACHE_DIR,
    step_duration=STEP_DURATION,
    refresh=False,
)
resolved_battery_variants = resolve_battery_variants(BATTERY_VARIANTS)
MAX_STEP = int(round(EPISODE_HOURS / STEP_DURATION))
processed_data.head()

## 3. Train the SB3 model

In [ ]:
model, eval_result = train_sb3_model_on_aemo(
    processed_data=processed_data,
    algorithm=SB3_ALGORITHM,
    battery_variants=resolved_battery_variants,
    episodes_per_variant=EPISODES_PER_VARIANT,
    max_step=MAX_STEP,
    step_duration=STEP_DURATION,
    action_mode=ACTION_MODE,
    degradation_mode=DEGRADATION_MODE,
    degradation_chemistry=DEGRADATION_CHEMISTRY,
    degradation_temperature=DEGRADATION_TEMPERATURE,
    random_episode_start=True,
    test_timesteps=TEST_TIMESTEPS,
    total_timesteps=TOTAL_TIMESTEPS,
    n_trials=N_TRIALS,
    n_jobs=N_JOBS,
    default_model=DEFAULT_MODEL,
)

MODEL_PATH = run_paths['output_dir'] / f'{SB3_ALGORITHM.lower()}_aemo_model.zip'
model.save(MODEL_PATH)
print(MODEL_PATH)
eval_result

## 4. Export rollout logs for evaluation or offline DT data

In [ ]:
rollout_env_fns = make_aemo_env_fns(
    processed_data=processed_data,
    battery_variants=resolved_battery_variants,
    episodes_per_variant=ROLLOUT_EPISODES_PER_VARIANT,
    max_step=MAX_STEP,
    step_duration=STEP_DURATION,
    action_mode=ACTION_MODE,
    degradation_mode=DEGRADATION_MODE,
    degradation_chemistry=DEGRADATION_CHEMISTRY,
    degradation_temperature=DEGRADATION_TEMPERATURE,
    random_episode_start=True,
)
rollout_vec_env = DummyVecEnv(rollout_env_fns)
try:
    episode_data = run_sb3_model_on_vec_env(model, rollout_vec_env, deterministic=DETERMINISTIC_ROLLOUT, max_steps=MAX_STEP)
finally:
    rollout_vec_env.close()

rollout_df = flatten_episode_data(episode_data)
rollout_path = run_paths['raw_dir'] / f'{SB3_ALGORITHM.lower()}_rollouts.parquet'
rollout_df.write_parquet(rollout_path)
print(rollout_path)
rollout_df.head()

## 5. Optional next step

Use the exported SB3 rollout parquet inside `aemo_simrun.ipynb` by adding it to the `BEHAVIOR_RUNS` config as an SB3 behavior source, or merge it into the DT dataset manually.